# UtaLearn — Ingestion PoC

Process a Japanese song into word-level aligned JSON + MP3 for the SyncEngine frontend.

**Pipeline:** YouTube → MP3 → WhisperX transcription → forced alignment (character-level) → morphological grouping (fugashi) → romaji (cutlet) → `song_data.json`

## 1. Install & GPU Check

In [ ]:
# whisperx --no-deps to keep Colab's torch; let pip freely resolve the rest
!pip install -q --no-deps git+https://github.com/m-bain/whisperX.git
!pip install -q git+https://github.com/pyannote/pyannote-audio.git faster-whisper ctranslate2 yt-dlp cutlet unidic-lite
# numpy was upgraded — realign scipy, then restart so new C extensions load
!pip install -q -U numpy scipy

# Restart runtime (packages persist on disk, but C extensions need a fresh process)
# After restart, skip this cell and run from "GPU Check" onward
import os
os.kill(os.getpid(), 9)

In [ ]:
import torch

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected. WhisperX will be very slow on CPU.")

## 2. Configuration

In [ ]:
import os

# --- Device ---
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
COMPUTE_TYPE = "float16" if DEVICE == "cuda" else "int8"

# --- WhisperX ---
WHISPER_MODEL = "large-v2"
LANGUAGE = "ja"

# --- Song ---
YOUTUBE_URL = "https://www.youtube.com/watch?v=xtfXl7TZTac"  # Yoru ni Kakeru - YOASOBI
SONG_TITLE = "Yoru ni Kakeru"
SONG_ARTIST = "YOASOBI"

# --- Paths ---
OUTPUT_DIR = "output"
os.makedirs(OUTPUT_DIR, exist_ok=True)
AUDIO_PATH = os.path.join(OUTPUT_DIR, "audio.mp3")
JSON_PATH = os.path.join(OUTPUT_DIR, "song_data.json")

# --- HuggingFace token (required for Japanese alignment model) ---
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
    print("HF_TOKEN loaded from Colab Secrets.")
except (ImportError, Exception):
    HF_TOKEN = os.environ.get("HF_TOKEN", "")
    if HF_TOKEN:
        print("HF_TOKEN loaded from environment variable.")
    else:
        print("WARNING: No HF_TOKEN found. Set it in Colab Secrets or as an env var.")
        print("The Japanese alignment model requires a HuggingFace token with access to:")
        print("  jonatasgrosman/wav2vec2-large-xlsr-53-japanese")

print(f"\nDevice: {DEVICE} | Compute: {COMPUTE_TYPE} | Model: {WHISPER_MODEL}")

## 3. Download Audio (yt-dlp)

In [ ]:
import subprocess
import glob as globmod

result = subprocess.run(
    [
        "yt-dlp",
        "-x", "--audio-format", "mp3", "--audio-quality", "0",
        "-o", os.path.join(OUTPUT_DIR, "audio.%(ext)s"),
        YOUTUBE_URL,
    ],
    capture_output=True,
    text=True,
)

if result.returncode != 0:
    print("STDERR:", result.stderr)
    raise RuntimeError("yt-dlp failed")

# yt-dlp manages the extension — find the actual file and rename to AUDIO_PATH
downloaded = globmod.glob(os.path.join(OUTPUT_DIR, "audio.*"))
print("Files in output:", downloaded)
if downloaded and downloaded[0] != AUDIO_PATH:
    os.rename(downloaded[0], AUDIO_PATH)

size_mb = os.path.getsize(AUDIO_PATH) / 1e6
print(f"Downloaded: {AUDIO_PATH} ({size_mb:.1f} MB)")

## 4. WhisperX Transcription

In [ ]:
import whisperx

# Very low VAD thresholds — singing over music needs aggressive detection
model = whisperx.load_model(
    WHISPER_MODEL, DEVICE, compute_type=COMPUTE_TYPE, language=LANGUAGE,
    vad_options={"vad_onset": 0.02, "vad_offset": 0.01},
)
audio = whisperx.load_audio(AUDIO_PATH)
result = model.transcribe(audio, batch_size=16, language=LANGUAGE)

print(f"Transcribed {len(result['segments'])} segments.\n")
for seg in result["segments"][:5]:
    print(f"  [{seg['start']:.2f} - {seg['end']:.2f}] {seg['text']}")

## 5. WhisperX Forced Alignment

In [ ]:
# Free transcription model VRAM
del model
torch.cuda.empty_cache()

align_model, align_metadata = whisperx.load_align_model(
    language_code=LANGUAGE, device=DEVICE, model_name="jonatasgrosman/wav2vec2-large-xlsr-53-japanese"
)
aligned = whisperx.align(
    result["segments"], align_model, align_metadata, audio, DEVICE,
    return_char_alignments=True,
)

# Free alignment model VRAM
del align_model
torch.cuda.empty_cache()

print(f"Aligned {len(aligned['segments'])} segments.\n")
# Preview character-level output for first segment
seg0 = aligned["segments"][0]
print(f"Segment 0 text: {seg0['text']}")
print(f"Segment 0 chars:")
for c in seg0.get("chars", [])[:10]:
    print(f"  {c}")

## 6. Character-to-Word Grouping + Romaji

WhisperX gives us per-character timestamps for Japanese. We use **fugashi** (MeCab) to tokenize text into morphemes, then greedily match each morpheme's characters against the WhisperX character timings to produce word-level timestamps. **Cutlet** converts each word to romaji.

In [ ]:
import cutlet

katsu = cutlet.Cutlet()
katsu.use_foreign_spelling = False  # Force transliteration for all text


def build_word_segments(segment: dict) -> list[dict]:
    """Group character-level alignments into word-level segments using morphological analysis."""
    text = segment.get("text", "")
    chars = segment.get("chars", [])

    if not text or not chars:
        return []

    # Tokenize into morphemes with fugashi (via cutlet's tagger)
    morphemes = []
    for token in katsu.tagger(text):
        surface = token.surface
        if not surface.strip():
            continue
        morphemes.append(surface)

    # Greedy-match morpheme characters to WhisperX char timings
    words = []
    char_idx = 0

    for morph in morphemes:
        morph_chars = list(morph)
        matched_timings = []

        for mc in morph_chars:
            # Advance past any chars not matching (whitespace, punctuation skipped by aligner)
            while char_idx < len(chars) and chars[char_idx].get("char", "") != mc:
                char_idx += 1
            if char_idx < len(chars):
                matched_timings.append(chars[char_idx])
                char_idx += 1

        if not matched_timings:
            continue

        # Collect start/end from matched characters
        starts = [t["start"] for t in matched_timings if "start" in t]
        ends = [t["end"] for t in matched_timings if "end" in t]
        scores = [t["score"] for t in matched_timings if "score" in t]

        word_entry = {
            "text": morph,
            "romaji": katsu.romaji(morph),
        }
        if starts:
            word_entry["start"] = min(starts)
        if ends:
            word_entry["end"] = max(ends)
        if scores:
            word_entry["score"] = round(sum(scores) / len(scores), 4)

        words.append(word_entry)

    return words


# Preview first segment
seg0 = aligned["segments"][0]
words0 = build_word_segments(seg0)
print(f"Segment 0: {seg0['text']}\n")
for w in words0:
    start = w.get('start', '?')
    end = w.get('end', '?')
    print(f"  [{start} - {end}] {w['text']} → {w['romaji']}  (score: {w.get('score', '?')})")

## 7. Export & Verification

In [ ]:
import json
from datetime import datetime, timezone

# Compute audio duration from loaded array (WhisperX uses 16kHz)
duration_seconds = round(len(audio) / 16000, 2)

# Build all segments
segments = []
for i, seg in enumerate(aligned["segments"]):
    words = build_word_segments(seg)
    # Build segment-level romaji from word romaji
    romaji = " ".join(w["romaji"] for w in words)

    segments.append({
        "id": i,
        "start": seg.get("start"),
        "end": seg.get("end"),
        "text": seg.get("text", ""),
        "romaji": romaji,
        "words": words,
    })

song_data = {
    "meta": {
        "title": SONG_TITLE,
        "artist": SONG_ARTIST,
        "youtube_url": YOUTUBE_URL,
        "language": LANGUAGE,
        "duration_seconds": duration_seconds,
        "generated_at": datetime.now(timezone.utc).isoformat(),
        "whisperx_model": WHISPER_MODEL,
        "align_model": "jonatasgrosman/wav2vec2-large-xlsr-53-japanese",
    },
    "segments": segments,
}

with open(JSON_PATH, "w", encoding="utf-8") as f:
    json.dump(song_data, f, ensure_ascii=False, indent=2)

print(f"Exported {len(segments)} segments to {JSON_PATH}")
print(f"Duration: {duration_seconds}s | JSON size: {os.path.getsize(JSON_PATH) / 1024:.1f} KB")

In [ ]:
# --- Validation ---
issues = []

for seg in segments:
    sid = seg["id"]

    # Segment-level checks
    if seg["start"] is None or seg["end"] is None:
        issues.append(f"Segment {sid}: missing segment-level timestamp")
    elif seg["start"] > seg["end"]:
        issues.append(f"Segment {sid}: inverted segment timestamps ({seg['start']} > {seg['end']})")

    # Word-level checks
    prev_end = None
    for j, w in enumerate(seg["words"]):
        if "start" not in w or "end" not in w:
            issues.append(f"Segment {sid}, word {j} ('{w['text']}'): missing timestamp")
            continue
        if w["start"] > w["end"]:
            issues.append(f"Segment {sid}, word {j} ('{w['text']}'): inverted timestamps")
        if prev_end is not None and w["start"] < prev_end - 0.01:  # 10ms tolerance
            issues.append(f"Segment {sid}, word {j} ('{w['text']}'): non-monotonic (start {w['start']:.3f} < prev end {prev_end:.3f})")
        prev_end = w.get("end", prev_end)

if issues:
    print(f"Found {len(issues)} issue(s):\n")
    for issue in issues[:20]:
        print(f"  - {issue}")
    if len(issues) > 20:
        print(f"  ... and {len(issues) - 20} more")
else:
    print("All checks passed.")

In [ ]:
# --- Preview first 5 segments ---
print("=" * 60)
for seg in segments[:5]:
    print(f"\n[{seg['start']} - {seg['end']}] Segment {seg['id']}")
    print(f"  JP:     {seg['text']}")
    print(f"  Romaji: {seg['romaji']}")
    print(f"  Words:  {len(seg['words'])}")
    for w in seg["words"][:6]:
        s = w.get('start', '?')
        e = w.get('end', '?')
        print(f"    [{s} - {e}] {w['text']} → {w['romaji']}")
    if len(seg["words"]) > 6:
        print(f"    ... +{len(seg['words']) - 6} more words")
print("\n" + "=" * 60)

In [ ]:
# --- Download files (Colab only) ---
try:
    from google.colab import files
    files.download(JSON_PATH)
    files.download(AUDIO_PATH)
    print("Downloads triggered.")
except ImportError:
    print(f"Not running in Colab. Files are at:")
    print(f"  {os.path.abspath(JSON_PATH)}")
    print(f"  {os.path.abspath(AUDIO_PATH)}")